In [6]:
from tests.common import gpt2_bytes_to_unicode

In [7]:
from tests.adapters import run_train_bpe

In [8]:
special_tokens = [
            "<|endoftext|>",
        ]

In [9]:
vocab, merges = run_train_bpe("tests/fixtures/tinystories_sample_5M.txt", 500, special_tokens)

In [10]:
from tests.test_tokenizer import get_tokenizer_from_vocab_merges_path
from tests.test_tokenizer import VOCAB_PATH, MERGES_PATH

In [13]:
tokenizer = get_tokenizer_from_vocab_merges_path(VOCAB_PATH, MERGES_PATH, special_tokens=["<|endoftext|>"])

In [14]:
tokenizer.encode("Hello, world!")

[15496, 11, 995, 0]

In [18]:
tokenizer.decode([15496, 11, 995, 0])

'Hello, world!'

In [20]:
tokenizer.encode("the cat ate")

[1169, 3797, 15063]

In [21]:
tokenizer.decode([1169, 3797, 15063])

'the cat ate'

In [23]:
import numpy as np

In [24]:
valid_data = np.load('artifacts/data/tinystories_valid.npy', mmap_mode="r") 

In [27]:
print(valid_data[:50])

[ 118  862  492  499  266  322  608  370  263  911  465   45  338 1965
 2594  349 1603  285 3147  534  391  888  328  263  390  477   47  339
  283  378  797  267  263 3147 1081  550  266 2555  336   47  316 4354
 4217  336  267  392 1098  474  888   47]


In [30]:
from cs336_basics.training import get_batch

In [31]:
x, y = get_batch(valid_data, 32, 256, "cpu")

In [34]:
print(x.shape, y.shape)

torch.Size([32, 256]) torch.Size([32, 256])


In [35]:
print(x)

tensor([[2665,   47, 7040,  ...,  600, 4624, 1476],
        [ 328,  406,   47,  ..., 1041, 1777,  975],
        [ 699,  267, 1744,  ...,   11,  410, 1190],
        ...,
        [ 870,   47,   11,  ...,  709,  266,  433],
        [ 384,  583,  283,  ...,  341,   47,  546],
        [ 411,  405,  378,  ..., 1071,  925,  375]])


In [43]:
vocab_size=10_000
context_length=256
d_model=512
num_layers=4
num_heads=16
d_ff=1344
rope_theta=10_000.0
device="cpu"

In [45]:
from cs336_basics.nn import TransformerLM

In [46]:
model = TransformerLM(
    vocab_size=vocab_size,
    context_length=context_length,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    d_ff=d_ff,
    rope_theta=rope_theta,
).to(device)

In [47]:
logits = model(x)

In [49]:
print(logits.shape)

torch.Size([32, 256, 10000])


In [48]:
print(logits)

tensor([[[-4.2015e-01, -2.8668e-01, -1.4364e-01,  ..., -4.6837e-02,
           4.6859e-02, -2.5784e-01],
         [ 1.7362e-01,  1.9468e-01,  9.0364e-02,  ..., -1.5536e-01,
          -5.4464e-02, -5.2513e-01],
         [-1.7725e-01, -1.1681e-01,  2.6802e-01,  ..., -4.6206e-02,
           1.4036e-02, -2.6592e-01],
         ...,
         [ 3.8137e-01, -4.0069e-01,  1.4271e-01,  ..., -3.5723e-01,
          -1.8746e-01, -3.9905e-02],
         [ 1.7287e-01,  1.8239e-02,  4.1502e-01,  ...,  1.3216e-01,
           1.1994e-01, -1.8222e-01],
         [ 8.3427e-03,  1.0240e-01, -4.6745e-02,  ..., -3.9703e-01,
           4.3098e-01,  2.1365e-01]],

        [[ 1.4545e-02,  5.4987e-01,  2.4542e-01,  ..., -4.0958e-01,
          -1.7207e-01, -4.8823e-01],
         [ 2.1473e-01,  7.8836e-01,  3.5010e-01,  ..., -2.4286e-02,
           5.3370e-01, -2.7973e-01],
         [ 1.4397e-01,  4.1092e-01,  2.2826e-01,  ..., -2.4472e-01,
          -2.5568e-03, -4.0542e-01],
         ...,
         [-4.9093e-02, -4